In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import seasons

In [2]:
# location of health export
filepath = '/Users/ryansponzilli/Developer/Python Projects/health-data/data/raw/apple_health/apple_health_export/export.xml'
# create element tree object
tree = ET.parse(filepath) 
# for every health record, extract the attributes
root = tree.getroot()
record_list = [x.attrib for x in root.iter('Record')]
workout_list = [x.attrib for x in root.iter('Workout')]

record_data = pd.DataFrame(record_list)
workout_data = pd.DataFrame(workout_list)

In [3]:
record_data.head()

,type,sourceName,sourceVersion,unit,creationDate,startDate,endDate,value,device
0,HKQuantityTypeIdentifierDietaryWater,WaterMinder,552,mL,2021-01-25 11:00:05 -0600,2021-01-25 11:00:04 -0600,2021-01-25 11:00:04 -0600,0,NaN
1,HKQuantityTypeIdentifierDietaryWater,WaterMinder,544,mL,2020-10-07 14:38:40 -0600,2020-10-07 14:38:40 -0600,2020-10-07 14:38:40 -0600,946.353,NaN
2,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 10:07:22 -0600,2020-08-14 10:07:18 -0600,2020-08-14 10:07:18 -0600,354.882,NaN
3,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 11:09:33 -0600,2020-08-14 11:09:28 -0600,2020-08-14 11:09:28 -0600,354.882,NaN
4,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 11:58:30 -0600,2020-08-14 11:58:30 -0600,2020-08-14 11:58:30 -0600,354.882,NaN


In [4]:
record_data_parsed = record_data.copy()

# proper type to dates
for col in ['creationDate', 'startDate', 'endDate']:
    record_data_parsed[col] = pd.to_datetime(record_data_parsed[col])

# get relevant dates
record_data_parsed = record_data_parsed.loc[(record_data_parsed['startDate'] >= pd.to_datetime(seasons.MIN_DATE, utc=True)) & (record_data_parsed['startDate'] <= pd.to_datetime(seasons.MAX_DATE, utc=True))]

# value is numeric, NaN if fails
# record_data_parsed['value_num'] = pd.to_numeric(record_data_parsed['value'], errors='coerce')

# some records do not measure anything, just count occurences
# filling with 1.0 (= one time) makes it easier to aggregate
record_data_parsed['value'] = record_data_parsed['value'].fillna(1.0)

# shorter observation names
record_data_parsed['type'] = record_data_parsed['type'].str.replace('HKQuantityTypeIdentifier', '')
record_data_parsed['type'] = record_data_parsed['type'].str.replace('HKCategoryTypeIdentifier', '')

# get relevant data
record_data_parsed = record_data_parsed.loc[record_data_parsed['type'].isin(["ActiveEnergyBurned", "HeartRate", "VO2Max", "RunningSpeed"])]
record_data_parsed = record_data_parsed[["type", "value", "unit", "creationDate", "startDate", "endDate"]].rename(columns={"creationDate": "creation_date", "startDate": "start_date", "endDate": "end_date"})
record_data_parsed = record_data_parsed.sort_values("start_date").reset_index(drop=True)

In [5]:
record_data_parsed

,type,value,unit,creation_date,start_date,end_date
0,ActiveEnergyBurned,0.317,Cal,2019-06-09 18:05:31-06:00,2019-06-09 18:00:19-06:00,2019-06-09 18:01:20-06:00
1,ActiveEnergyBurned,0.287,Cal,2019-06-09 18:05:31-06:00,2019-06-09 18:01:20-06:00,2019-06-09 18:02:22-06:00
2,ActiveEnergyBurned,0.064,Cal,2019-06-09 18:05:31-06:00,2019-06-09 18:02:52-06:00,2019-06-09 18:03:03-06:00
3,ActiveEnergyBurned,0.21,Cal,2019-06-09 18:06:36-06:00,2019-06-09 18:03:03-06:00,2019-06-09 18:04:04-06:00
4,HeartRate,68,count/min,2019-06-09 18:05:31-06:00,2019-06-09 18:03:07-06:00,2019-06-09 18:03:07-06:00
...,...,...,...,...,...,...
1480836,ActiveEnergyBurned,0.63,Cal,2023-05-15 15:55:13-06:00,2023-05-15 15:54:03-06:00,2023-05-15 15:54:44-06:00
1480837,ActiveEnergyBurned,0.747,Cal,2023-05-15 15:56:34-06:00,2023-05-15 15:55:14-06:00,2023-05-15 15:56:16-06:00
1480838,ActiveEnergyBurned,1.073,Cal,2023-05-15 15:58:06-06:00,2023-05-15 15:56:16-06:00,2023-05-15 15:57:17-06:00
1480839,ActiveEnergyBurned,1.298,Cal,2023-05-15 15:59:27-06:00,2023-05-15 15:57:17-06:00,2023-05-15 15:58:18-06:00


In [6]:
workout_data.head()

,workoutActivityType,duration,durationUnit,sourceName,sourceVersion,creationDate,startDate,endDate,device
0,HKWorkoutActivityTypeCycling,17.29111891587575,min,Ryan’s Apple Watch,4.0,2017-10-25 11:05:28 -0600,2017-10-25 10:48:09 -0600,2017-10-25 11:05:27 -0600,NaN
1,HKWorkoutActivityTypeCycling,14.53230218291283,min,Ryan’s Apple Watch,4.1,2017-11-15 12:05:24 -0600,2017-11-15 11:50:51 -0600,2017-11-15 12:05:23 -0600,NaN
2,HKWorkoutActivityTypeRunning,41.06247799992561,min,Ryan’s Apple Watch,4.3.1,2018-06-20 19:04:16 -0600,2018-06-20 18:09:09 -0600,2018-06-20 19:04:14 -0600,NaN
3,HKWorkoutActivityTypeRunning,51.96426715056101,min,Ryan’s Apple Watch,4.3.1,2018-07-04 06:26:07 -0600,2018-07-04 05:26:57 -0600,2018-07-04 06:26:03 -0600,NaN
4,HKWorkoutActivityTypeCycling,37.01337313254674,min,Ryan’s Apple Watch,4.3.2,2018-07-18 17:35:16 -0600,2018-07-18 16:58:13 -0600,2018-07-18 17:35:14 -0600,NaN


In [59]:
workout_data_parsed = workout_data.copy()

# proper type to dates
for col in ['creationDate', 'startDate', 'endDate']:
    workout_data_parsed[col] = pd.to_datetime(workout_data_parsed[col])

workout_data_parsed['duration'] = workout_data_parsed['duration'].astype(float)

# get relevant dates
workout_data_parsed = workout_data_parsed.loc[(workout_data_parsed['startDate'] >= pd.to_datetime(seasons.MIN_DATE, utc=True)) & (workout_data_parsed['startDate'] <= pd.to_datetime(seasons.MAX_DATE, utc=True))]

# get relevant data
# workout_data_parsed = workout_data_parsed.loc[workout_data_parsed['workoutActivityType'] == "HKWorkoutActivityTypeRunning"]
# workout_data_parsed = workout_data_parsed.loc[workout_data_parsed['sourceName'] != "Strava"]
workout_data_parsed = workout_data_parsed[["workoutActivityType", "creationDate", "startDate", "endDate", "duration", "durationUnit"]].rename(columns={"creationDate": "creation_datetime", "startDate": "start_datetime", "endDate": "end_datetime"})
workout_data_parsed = workout_data_parsed.sort_values("start_datetime").reset_index(drop=True)

In [60]:
workout_data_parsed

,workoutActivityType,creation_datetime,start_datetime,end_datetime,duration,durationUnit
0,HKWorkoutActivityTypeRunning,2019-06-10 06:46:26-06:00,2019-06-10 05:57:37-06:00,2019-06-10 06:46:11-06:00,48.564544,min
1,HKWorkoutActivityTypeRunning,2019-06-11 06:32:34-06:00,2019-06-11 05:58:59-06:00,2019-06-11 06:32:25-06:00,33.435890,min
2,HKWorkoutActivityTypeRunning,2019-06-12 06:35:28-06:00,2019-06-12 05:53:26-06:00,2019-06-12 06:35:17-06:00,31.116127,min
3,HKWorkoutActivityTypeRunning,2019-06-13 06:40:16-06:00,2019-06-13 05:54:03-06:00,2019-06-13 06:40:04-06:00,46.021483,min
4,HKWorkoutActivityTypeCycling,2019-06-14 13:56:26-06:00,2019-06-14 10:00:41-06:00,2019-06-14 13:56:10-06:00,60.731609,min
...,...,...,...,...,...,...
1291,HKWorkoutActivityTypeRunning,2023-05-04 14:46:16-06:00,2023-05-04 14:24:15-06:00,2023-05-04 14:42:07-06:00,5.321514,min
1292,HKWorkoutActivityTypeRunning,2023-05-05 14:11:58-06:00,2023-05-05 13:44:32-06:00,2023-05-05 14:11:46-06:00,25.532650,min
1293,HKWorkoutActivityTypeRunning,2023-05-06 10:15:51-06:00,2023-05-06 09:58:39-06:00,2023-05-06 10:15:17-06:00,15.347009,min
1294,HKWorkoutActivityTypeRunning,2023-05-07 11:57:58-06:00,2023-05-06 11:45:59-06:00,2023-05-06 11:50:51-06:00,4.866667,min


In [53]:
gdf = gpd.read_parquet("../data/track_info.parquet")
gdf["track_file_datetime"] = pd.to_datetime(gdf["track_file_datetime"], utc=True)
gdf

,track_file,track_file_datetime,track_file_date,start_datetime,end_datetime,year,season,total_duration,total_distance (mi),average_pace (min/mi),total_elevation_change (m),in_stc,start_point,end_point
0,/Users/ryansponzilli/Developer/Python Projects...,2019-06-10 07:46:00+00:00,2019-06-10,2019-06-10 11:57:42+00:00,2019-06-10 12:46:10+00:00,Freshman,Summer Training,0 days 00:48:28,3.063156,8.542777,77.126862,True,POINT (-88.32154 41.91101),POINT (-88.32166 41.9109)
1,/Users/ryansponzilli/Developer/Python Projects...,2019-06-11 07:32:00+00:00,2019-06-11,2019-06-11 11:59:03+00:00,2019-06-11 12:32:23+00:00,Freshman,Summer Training,0 days 00:33:20,3.172008,8.767494,122.178753,True,POINT (-88.34764 41.928),POINT (-88.34792 41.92783)
2,/Users/ryansponzilli/Developer/Python Projects...,2019-06-12 07:35:00+00:00,2019-06-12,2019-06-12 11:53:30+00:00,2019-06-12 12:35:13+00:00,Freshman,Summer Training,0 days 00:41:43,3.052702,8.581829,132.038013,True,POINT (-88.32152 41.91252),POINT (-88.32172 41.911)
3,/Users/ryansponzilli/Developer/Python Projects...,2019-06-14 14:56:00+00:00,2019-06-14,2019-06-14 16:00:46+00:00,2019-06-14 19:56:09+00:00,Freshman,Summer Training,0 days 03:55:23,5.988500,7.503064,121.541945,True,POINT (-88.28753 41.90455),POINT (-88.28622 41.90465)
4,/Users/ryansponzilli/Developer/Python Projects...,2019-06-14 17:15:00+00:00,2019-06-14,2019-06-14 20:59:18+00:00,2019-06-14 22:15:01+00:00,Freshman,Summer Training,0 days 01:15:43,3.664106,6.928237,87.290450,True,POINT (-88.27906 41.90787),POINT (-88.28638 41.90464)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123,/Users/ryansponzilli/Developer/Python Projects...,2023-05-04 15:10:00+00:00,2023-05-04,2023-05-04 19:54:31+00:00,2023-05-04 20:10:21+00:00,Senior,Track Season,0 days 00:15:50,2.032107,7.547855,23.622665,True,POINT (-88.27989 41.92867),POINT (-88.27976 41.92838)
1124,/Users/ryansponzilli/Developer/Python Projects...,2023-05-04 15:42:00+00:00,2023-05-04,2023-05-04 20:24:15+00:00,2023-05-04 20:42:07+00:00,Senior,Track Season,0 days 00:17:52,0.999846,5.064146,8.642043,True,POINT (-88.27658 41.92845),POINT (-88.27658 41.92851)
1125,/Users/ryansponzilli/Developer/Python Projects...,2023-05-05 15:11:00+00:00,2023-05-05,2023-05-05 19:44:31+00:00,2023-05-05 20:11:23+00:00,Senior,Track Season,0 days 00:26:52,3.509378,6.976707,43.772437,True,POINT (-88.27988 41.92846),POINT (-88.27952 41.9282)
1126,/Users/ryansponzilli/Developer/Python Projects...,2023-05-06 11:15:00+00:00,2023-05-06,2023-05-06 15:58:42+00:00,2023-05-06 16:15:16+00:00,Senior,Track Season,0 days 00:16:34,1.985944,7.678578,38.987511,True,POINT (-88.32198 41.95005),POINT (-88.32167 41.95012)


I want to match each gpx track to a workout_data_parsed row

In [61]:
# match each gpx track to a row in parsed_workout_data
merge_cols = []
for i, row in gdf.iterrows():
    closest_match_idx = np.abs(workout_data_parsed['start_datetime'] - row["start_datetime"]).argmin()
    merge_cols.append(workout_data_parsed.iloc[closest_match_idx]["start_datetime"])
gdf["merge_col"] = merge_cols

In [62]:
gdf['merge_col_diff'] = np.abs(gdf['merge_col'] - gdf['start_datetime'])
gdf = gdf.sort_values(["track_file_date", "merge_col_diff"])
gdf.loc[gdf['merge_col'].duplicated(), 'merge_col'] = None

In [ ]:
# merge to get workout type
new = gdf.merge(workout_data_parsed[["start_datetime", "workoutActivityType"]], how='left', left_on="merge_col", right_on="start_datetime", suffixes=[None, "_y"])
# exclude all non-running workouts
new = new.loc[new["workoutActivityType"].isin(["HKWorkoutActivityTypeRunning", None])].drop(columns=["merge_col", "merge_col_diff", "start_datetime_y", "workoutActivityType"])
new

,track_file,track_file_datetime,track_file_date,start_datetime,end_datetime,year,season,total_duration,total_distance (mi),average_pace (min/mi),total_elevation_change (m),in_stc,start_point,end_point,merge_col,merge_col_diff
0,/Users/ryansponzilli/Developer/Python Projects...,2019-06-10 07:46:00+00:00,2019-06-10,2019-06-10 11:57:42+00:00,2019-06-10 12:46:10+00:00,Freshman,Summer Training,0 days 00:48:28,3.063156,8.542777,77.126862,True,POINT (-88.32154 41.91101),POINT (-88.32166 41.9109),2019-06-10 05:57:37-06:00,0 days 00:00:05
1,/Users/ryansponzilli/Developer/Python Projects...,2019-06-11 07:32:00+00:00,2019-06-11,2019-06-11 11:59:03+00:00,2019-06-11 12:32:23+00:00,Freshman,Summer Training,0 days 00:33:20,3.172008,8.767494,122.178753,True,POINT (-88.34764 41.928),POINT (-88.34792 41.92783),2019-06-11 05:58:59-06:00,0 days 00:00:04
2,/Users/ryansponzilli/Developer/Python Projects...,2019-06-12 07:35:00+00:00,2019-06-12,2019-06-12 11:53:30+00:00,2019-06-12 12:35:13+00:00,Freshman,Summer Training,0 days 00:41:43,3.052702,8.581829,132.038013,True,POINT (-88.32152 41.91252),POINT (-88.32172 41.911),2019-06-12 05:53:26-06:00,0 days 00:00:04
5,/Users/ryansponzilli/Developer/Python Projects...,2019-06-17 07:22:00+00:00,2019-06-17,2019-06-17 11:52:32+00:00,2019-06-17 12:22:32+00:00,Freshman,Summer Training,0 days 00:30:00,2.983042,9.088904,86.000490,True,POINT (-88.32145 41.911),POINT (-88.32154 41.91096),2019-06-17 05:52:28-06:00,0 days 00:00:04
6,/Users/ryansponzilli/Developer/Python Projects...,2019-06-18 07:19:00+00:00,2019-06-18,2019-06-18 11:53:58+00:00,2019-06-18 12:19:21+00:00,Freshman,Summer Training,0 days 00:25:23,2.355063,9.380388,95.971917,True,POINT (-88.35416 41.92575),POINT (-88.34797 41.92783),2019-06-18 05:50:34-06:00,0 days 00:03:24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,/Users/ryansponzilli/Developer/Python Projects...,2023-05-04 15:42:00+00:00,2023-05-04,2023-05-04 20:24:15+00:00,2023-05-04 20:42:07+00:00,Senior,Track Season,0 days 00:17:52,0.999846,5.064146,8.642043,True,POINT (-88.27658 41.92845),POINT (-88.27658 41.92851),2023-05-04 14:24:15-06:00,0 days 00:00:00
1100,/Users/ryansponzilli/Developer/Python Projects...,2023-05-04 15:10:00+00:00,2023-05-04,2023-05-04 19:54:31+00:00,2023-05-04 20:10:21+00:00,Senior,Track Season,0 days 00:15:50,2.032107,7.547855,23.622665,True,POINT (-88.27989 41.92867),POINT (-88.27976 41.92838),2023-05-04 13:54:27-06:00,0 days 00:00:04
1101,/Users/ryansponzilli/Developer/Python Projects...,2023-05-05 15:11:00+00:00,2023-05-05,2023-05-05 19:44:31+00:00,2023-05-05 20:11:23+00:00,Senior,Track Season,0 days 00:26:52,3.509378,6.976707,43.772437,True,POINT (-88.27988 41.92846),POINT (-88.27952 41.9282),2023-05-05 13:44:32-06:00,0 days 00:00:01
1102,/Users/ryansponzilli/Developer/Python Projects...,2023-05-06 11:15:00+00:00,2023-05-06,2023-05-06 15:58:42+00:00,2023-05-06 16:15:16+00:00,Senior,Track Season,0 days 00:16:34,1.985944,7.678578,38.987511,True,POINT (-88.32198 41.95005),POINT (-88.32167 41.95012),2023-05-06 09:58:39-06:00,0 days 00:00:03
